# Fine-tune Qwen2.5-3B on Facebook Messages

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `train_train.jsonl` and `train_val.jsonl` using the Files panel (folder icon on the left)
3. Run cells top to bottom

In [ ]:
# Cell 1 — Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — change runtime to T4')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "transformers>=4.45.0" "trl>=0.12.0" "peft>=0.13.0" "accelerate>=1.0.0" "bitsandbytes>=0.44.0" "datasets>=3.0.0" "huggingface_hub>=0.25.0"
print('Done!')

In [ ]:
# Cell 3 — Config (edit these)
BASE_MODEL      = 'unsloth/Qwen2.5-3B-Instruct'
TRAIN_FILE      = '/content/train_train.jsonl'   # path after uploading
VAL_FILE        = '/content/train_val.jsonl'
OUTPUT_DIR      = '/content/output'
NUM_EPOCHS      = 3
BATCH_SIZE      = 4
GRAD_ACCUM      = 4
LEARNING_RATE   = 2e-4
MAX_SEQ_LENGTH  = 1024
LORA_RANK       = 32

# HuggingFace Hub — fill in to auto-upload after training
# Leave HF_TOKEN empty to skip hub upload
HF_TOKEN        = ''   # 'hf_xxxxxxxx'
HUB_MODEL_ID    = 'itsrenzosamaaa/robby-chat-model'  # your HF username/model-name

In [ ]:
# Cell 4 — Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(model.print_trainable_parameters())

In [ ]:
# Cell 5 — Load dataset
import json
from datasets import Dataset

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f'Loaded {len(records)} examples from {path}')
    return Dataset.from_list(records)

def format_prompt(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_ds = load_jsonl(TRAIN_FILE).map(format_prompt)
val_ds   = load_jsonl(VAL_FILE).map(format_prompt)

In [ ]:
# Cell 6 — Train
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_steps=20,
        logging_steps=10,
        save_steps=50,
        eval_strategy='steps',
        eval_steps=50,
        save_total_limit=2,
        bf16=False,
        fp16=True,   # T4 uses fp16, not bf16
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        report_to='none',
    ),
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 7 — Save merged model
merged_path = OUTPUT_DIR + '/merged_model'
print(f'Saving merged model to {merged_path}...')
model.save_pretrained_merged(merged_path, tokenizer, save_method='merged_16bit')
print('Saved!')

In [ ]:
# Cell 8a — Push to HuggingFace Hub (run this if you set HF_TOKEN above)
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub_merged(
        HUB_MODEL_ID,
        tokenizer,
        save_method='lora',
        token=HF_TOKEN,
    )
    print(f'Pushed to: huggingface.co/{HUB_MODEL_ID}')
else:
    print('HF_TOKEN not set — skipping hub upload. Download the zip in Cell 8b instead.')

In [ ]:
# Cell 8b — Download as ZIP (alternative if not using HuggingFace)
# This zips the model and lets you download it directly from Colab
import shutil
zip_path = '/content/robby-model'
shutil.make_archive(zip_path, 'zip', merged_path)
print(f'Zip created: {zip_path}.zip')
print('Go to Files panel (left sidebar) → right-click robby-model.zip → Download')